# Phase 4 - Latent Feature Extraction

This notebook extracts 16-dimensional latent features from the trained Autoencoder encoder. The production logic stays in `src/latent_extraction.py`; this notebook is only an interactive wrapper.

In [ ]:
# Cell 1 - Set up imports and make the project package importable from notebooks.
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
print(f"Project root: {PROJECT_ROOT}")
print(f"Config path: {CONFIG_PATH}")


In [ ]:
# Cell 2 - Load the shared YAML config and resolve Phase 4 artifact paths.
from src.data_loading import load_config, resolve_project_path

config = load_config(CONFIG_PATH)
processed_dir = resolve_project_path(config["paths"]["processed_dir"], PROJECT_ROOT)
models_dir = resolve_project_path(config["paths"]["models_dir"], PROJECT_ROOT)
metrics_dir = resolve_project_path(config["paths"]["metrics_dir"], PROJECT_ROOT)

print(f"Input feature count: {config['autoencoder']['input_dim']}")
print(f"Latent feature count: {config['autoencoder']['latent_dim']}")


In [ ]:
# Cell 3 - Confirm that Phase 2 arrays and the Phase 3 encoder checkpoint are available.
required_inputs = [
    processed_dir / "X_train.npy",
    processed_dir / "X_test.npy",
    processed_dir / "y_train.npy",
    processed_dir / "y_test.npy",
    models_dir / "encoder.pt",
]

missing_inputs = [path for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"Missing required artifacts: {missing_inputs}")

for path in required_inputs:
    print(f"Found: {path}")


In [ ]:
# Cell 4 - Run latent feature extraction only if outputs are missing, or force regeneration.
from src.latent_extraction import extract_latent_features

FORCE_REEXTRACT = False
required_outputs = [
    processed_dir / "Z_train.npy",
    processed_dir / "Z_test.npy",
    metrics_dir / "latent_extraction_report.json",
]

if FORCE_REEXTRACT or not all(path.exists() for path in required_outputs):
    latent_report = extract_latent_features(CONFIG_PATH, device_name="auto")
else:
    with open(metrics_dir / "latent_extraction_report.json", "r", encoding="utf-8") as file:
        latent_report = json.load(file)

print(json.dumps(latent_report, indent=2))


In [ ]:
# Cell 5 - Load latent arrays as memory maps and verify their shapes without loading all rows into RAM.
z_train = np.load(processed_dir / "Z_train.npy", mmap_mode="r")
z_test = np.load(processed_dir / "Z_test.npy", mmap_mode="r")
y_train = np.load(processed_dir / "y_train.npy", mmap_mode="r")
y_test = np.load(processed_dir / "y_test.npy", mmap_mode="r")

latent_shape_summary = pd.DataFrame([
    {"array": "Z_train", "shape": tuple(z_train.shape), "dtype": str(z_train.dtype)},
    {"array": "Z_test", "shape": tuple(z_test.shape), "dtype": str(z_test.dtype)},
    {"array": "y_train", "shape": tuple(y_train.shape), "dtype": str(y_train.dtype)},
    {"array": "y_test", "shape": tuple(y_test.shape), "dtype": str(y_test.dtype)},
])
latent_shape_summary


In [ ]:
# Cell 6 - Check latent value ranges in batches; ReLU latent activation should make all values non-negative.
def summarize_memmap(name, array, step=250_000):
    min_value = float("inf")
    max_value = float("-inf")
    finite = True
    for start in range(0, array.shape[0], step):
        block = array[start : start + step]
        finite = finite and bool(np.isfinite(block).all())
        min_value = min(min_value, float(block.min()))
        max_value = max(max_value, float(block.max()))
    return {"array": name, "finite": finite, "min": min_value, "max": max_value}

range_summary = pd.DataFrame([
    summarize_memmap("Z_train", z_train),
    summarize_memmap("Z_test", z_test),
])
range_summary


In [ ]:
# Cell 7 - Preview the first few latent feature rows for manual inspection.
latent_columns = [f"z_{index:02d}" for index in range(z_train.shape[1])]
pd.DataFrame(np.array(z_train[:5]), columns=latent_columns)
